# 📚 Book Recommendation System — Collaborative Filtering

This notebook builds a book recommendation engine using three collaborative
filtering approaches:

1. **User-Based CF** — recommend books liked by similar users
2. **Item-Based CF** — recommend books similar to ones you already liked
3. **Matrix Factorization (SVD)** — learn latent taste/genre factors and predict ratings

It runs end-to-end on synthetic sample data (no downloads needed), and can be
pointed at a real dataset (e.g. Book-Crossing, Goodbooks-10k) by swapping out
the data-loading cell.

**Sections:**
- 1. Setup & Imports
- 2. Load Data (synthetic, or plug in your own CSVs)
- 3. Build the User–Item Rating Matrix
- 4. User-Based Collaborative Filtering
- 5. Item-Based Collaborative Filtering
- 6. Matrix Factorization with SVD
- 7. Model Evaluation (RMSE)
- 8. Try It Yourself


## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

pd.set_option("display.max_columns", 10)
np.random.seed(42)


## 2. Load Data

`load_sample_data()` generates a small synthetic dataset (users, books,
ratings) so the notebook works immediately with no external files.

**To use real data instead:** replace this cell with something like:

```python
books = pd.read_csv("books.csv")      # columns: book_id, title, author, genre
ratings = pd.read_csv("ratings.csv")  # columns: user_id, book_id, rating
```

Popular public datasets that fit this format: the **Book-Crossing** dataset
and the **Goodbooks-10k** dataset.


In [ ]:
def load_sample_data(n_users=40, n_books=25, n_ratings=400, seed=42):
    """Generates a small synthetic ratings dataset so the notebook
    runs end-to-end with no external files. Swap this out for your
    own ratings.csv (columns: user_id, book_id, rating) and
    books.csv (columns: book_id, title, author, genre) when you
    have real data."""
    rng = np.random.default_rng(seed)

    genres = ["Fantasy", "Mystery", "Romance", "Sci-Fi", "Non-Fiction"]
    titles = [
        "The Silent Forest", "Shadows of Time", "Love in Autumn",
        "The Last Algorithm", "Whispering Stars", "Blood Moon Rising",
        "The Paper Kingdom", "Beyond the Horizon", "Echoes of War",
        "The Quiet Storm", "City of Glass", "Winter's Promise",
        "The Forgotten Path", "Rise of the Machines", "Golden Hour",
        "The Ocean's Secret", "Midnight Library", "The Iron Crown",
        "Letters to Nowhere", "The Glass House", "Fire and Ash",
        "The Wandering Mind", "Song of the Sea", "Broken Compass",
        "The Last Chapter",
    ]

    books = pd.DataFrame({
        "book_id": range(1, n_books + 1),
        "title": titles[:n_books],
        "author": [f"Author {chr(65 + i % 20)}" for i in range(n_books)],
        "genre": [genres[i % len(genres)] for i in range(n_books)],
    })

    # Give each user a genre preference so ratings aren't pure noise
    user_pref = rng.choice(genres, size=n_users)
    rows = []
    seen = set()
    while len(rows) < n_ratings:
        u = rng.integers(1, n_users + 1)
        b = rng.integers(1, n_books + 1)
        if (u, b) in seen:
            continue
        seen.add((u, b))
        base = 3.0
        if books.loc[b - 1, "genre"] == user_pref[u - 1]:
            base += 1.5
        rating = np.clip(rng.normal(base, 0.9), 1, 5)
        rows.append((u, b, round(rating)))

    ratings = pd.DataFrame(rows, columns=["user_id", "book_id", "rating"])
    return books, ratings


N_USERS, N_BOOKS = 40, 25
books, ratings = load_sample_data(N_USERS, N_BOOKS)

print(f"{len(books)} books, {len(ratings)} ratings from {N_USERS} users\n")
books.head()


In [ ]:
ratings.head()

## 3. Build the User–Item Rating Matrix

Rows = users, columns = books, cells = ratings (NaN where a user hasn't
rated that book). Every CF method below is built on top of this matrix.


In [ ]:
def build_matrix(ratings, n_users, n_books):
    matrix = ratings.pivot_table(
        index="user_id", columns="book_id", values="rating"
    ).reindex(index=range(1, n_users + 1), columns=range(1, n_books + 1))
    return matrix


matrix = build_matrix(ratings, N_USERS, N_BOOKS)
matrix.iloc[:8, :8]


## 4. User-Based Collaborative Filtering

Idea: find users whose rating patterns are similar to the target user
(via **cosine similarity**), then recommend books those similar users
rated highly.


In [ ]:
def _attach_titles(ranked, books):
    out = []
    for book_id, score in ranked:
        title = books.loc[books.book_id == book_id, "title"].values[0]
        genre = books.loc[books.book_id == book_id, "genre"].values[0]
        out.append((book_id, title, genre, round(float(score), 2)))
    return out


def user_based_recommend(matrix, user_id, books, top_n=5, k_neighbors=10):
    filled = matrix.fillna(0)
    sim = cosine_similarity(filled)
    sim_df = pd.DataFrame(sim, index=matrix.index, columns=matrix.index)

    neighbors = sim_df[user_id].drop(user_id).sort_values(ascending=False)
    neighbors = neighbors[neighbors > 0].head(k_neighbors)

    user_rated = matrix.loc[user_id].dropna().index
    scores = {}
    for book_id in matrix.columns:
        if book_id in user_rated:
            continue
        num, den = 0.0, 0.0
        for neighbor_id, similarity in neighbors.items():
            r = matrix.loc[neighbor_id, book_id]
            if not np.isnan(r):
                num += similarity * r
                den += similarity
        if den > 0:
            scores[book_id] = num / den

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return _attach_titles(ranked, books)


demo_user = 1
pd.DataFrame(user_based_recommend(matrix, demo_user, books),
             columns=["book_id", "title", "genre", "predicted_rating"])


## 5. Item-Based Collaborative Filtering

Idea: find books similar to the ones the user already rated highly
(similarity computed across how *all users* rated each book pair).
This is the approach Amazon popularized ("customers who liked this also liked...").


In [ ]:
def item_based_recommend(matrix, user_id, books, top_n=5):
    filled = matrix.fillna(0)
    item_sim = cosine_similarity(filled.T)
    item_sim_df = pd.DataFrame(item_sim, index=matrix.columns, columns=matrix.columns)

    user_ratings = matrix.loc[user_id].dropna()
    scores = {}
    for book_id in matrix.columns:
        if book_id in user_ratings.index:
            continue
        sims = item_sim_df[book_id][user_ratings.index]
        num = (sims * user_ratings).sum()
        den = sims.abs().sum()
        if den > 0:
            scores[book_id] = num / den

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return _attach_titles(ranked, books)


pd.DataFrame(item_based_recommend(matrix, demo_user, books),
             columns=["book_id", "title", "genre", "predicted_rating"])


## 6. Matrix Factorization with SVD

Idea: decompose the sparse rating matrix into low-rank **user factors**
and **item factors** (latent "taste dimensions"). Multiplying them back
together fills in predicted ratings for books the user hasn't rated yet.
This is the technique that won the Netflix Prize.


In [ ]:
def svd_recommend(matrix, user_id, books, top_n=5, n_components=8):
    filled = matrix.fillna(matrix.stack().mean())
    n_components = min(n_components, min(filled.shape) - 1)

    svd = TruncatedSVD(n_components=n_components, random_state=42)
    user_factors = svd.fit_transform(filled)
    item_factors = svd.components_

    predicted = pd.DataFrame(
        user_factors @ item_factors, index=matrix.index, columns=matrix.columns
    )

    already_rated = matrix.loc[user_id].dropna().index
    preds = predicted.loc[user_id].drop(already_rated)
    ranked = preds.sort_values(ascending=False).head(top_n)
    return _attach_titles(list(ranked.items()), books)


pd.DataFrame(svd_recommend(matrix, demo_user, books),
             columns=["book_id", "title", "genre", "predicted_rating"])


## 7. Model Evaluation (RMSE)

We hold out 20% of ratings, train the SVD model on the rest, and measure
how far off its predictions are on the held-out ratings using **RMSE**
(lower is better; a random 1–5 guesser scores ~1.6-2.0).


In [ ]:
def evaluate_svd(ratings, n_users, n_books, n_components=8, test_size=0.2):
    train, test = train_test_split(ratings, test_size=test_size, random_state=42)

    train_matrix = build_matrix(train, n_users, n_books)
    global_mean = train_matrix.stack().mean()
    filled = train_matrix.fillna(global_mean)

    nc = min(n_components, min(filled.shape) - 1)
    svd = TruncatedSVD(n_components=nc, random_state=42)
    user_factors = svd.fit_transform(filled)
    item_factors = svd.components_
    predicted = pd.DataFrame(
        user_factors @ item_factors, index=train_matrix.index, columns=train_matrix.columns
    )

    y_true, y_pred = [], []
    for _, row in test.iterrows():
        if row.user_id in predicted.index and row.book_id in predicted.columns:
            y_true.append(row.rating)
            y_pred.append(np.clip(predicted.loc[row.user_id, row.book_id], 1, 5))

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return rmse


rmse = evaluate_svd(ratings, N_USERS, N_BOOKS)
print(f"SVD model RMSE on held-out test ratings: {rmse:.3f}")


## 8. Try It Yourself

Change `demo_user` below to any user_id from 1–40 and compare what each
method recommends.


In [ ]:
demo_user = 7  # try changing this

print("Books this user already rated:")
display(matrix.loc[demo_user].dropna())

print("\nUser-Based CF:")
display(pd.DataFrame(user_based_recommend(matrix, demo_user, books),
                      columns=["book_id", "title", "genre", "predicted_rating"]))

print("\nItem-Based CF:")
display(pd.DataFrame(item_based_recommend(matrix, demo_user, books),
                      columns=["book_id", "title", "genre", "predicted_rating"]))

print("\nSVD (Matrix Factorization):")
display(pd.DataFrame(svd_recommend(matrix, demo_user, books),
                      columns=["book_id", "title", "genre", "predicted_rating"]))
